In [ ]:
# Question - 1 : Implement a FAISS-based Retrieval System that fetches the most relevant document for query-based retrieval.

In [1]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


model = SentenceTransformer("all-MiniLM-L6-v2")
docs = [
    "Artificial Intelligence is transforming the tech industry.",
    "Python is a popular programming language for data science.",
    "FAISS is a library for efficient similarity search."
]


document_embeddings = model.encode(docs)

document_embeddings = np.array(document_embeddings, dtype=np.float32)

dimension = document_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(document_embeddings)


query = ["What is FAISS used for?"]
query_embedding = model.encode(query)
query_embedding = np.array(query_embedding, dtype=np.float32)


distances, indices = index.search(query_embedding, k=1)


best_match_index = indices[0][0]
print("Query:", query[0])
print("Most Relevant Document:", docs[best_match_index])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: What is FAISS used for?
Most Relevant Document: FAISS is a library for efficient similarity search.


In [2]:
# !pip install google-genai python-dotenv

In [ ]:
# Question - 2 : Use Gemini API to generate answers based on retrieved knowledge.

In [3]:
import os
from dotenv import load_dotenv
from google import genai

# 1. .env
load_dotenv()


client = genai.Client()

docs = [
    "Doc 1: Retrieval-Augmented Generation (RAG) reduces LLM hallucinations by providing external domain-specific context.",
    "Doc 2: RAG allows models to ground their responses in factual data without retraining the whole model.Give me simple content text."
]

# 3. Context & Prompt
context = "\n".join(docs)
user_question = "How does RAG improve accuracy?"

prompt = f"""Context:
{context}

Question: {user_question}"""

# 4. Gemini API-ai call 
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)


print(response.text)

Based on the context provided, RAG improves accuracy in two main ways:

1. **Provides External Context:** It supplies external, domain-specific information to the model, which reduces hallucinations (false information).
2. **Grounds Responses in Facts:** It allows the model to base its answers on real, factual data without needing to retrain the entire model.


In [ ]:
# Question - 3 : Implement FAISS with a hybrid (dense & sparse) vector search for improving information retrieval.

In [4]:
#!pip install faiss-cpu sentence-transformers rank_bm25 numpy

In [5]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# 1. Documents & Model initialization
docs = [
    "Machine learning and AI", 
    "FAISS vector search indexing", 
    "Python database and retrieval"
]
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Dense Index (FAISS) setup
dense_embeddings = model.encode(docs).astype('float32')
dimension = dense_embeddings.shape[1]
dense_index = faiss.IndexFlatL2(dimension)
dense_index.add(dense_embeddings)

# 3. Sparse Index (BM25) setup
tokenized_docs = [d.lower().split() for d in docs]
bm25 = BM25Okapi(tokenized_docs)

# 4. Query processing
query = "FAISS search"
query_embedding = model.encode([query]).astype('float32')

# Get top-k results from both (e.g., k=2)
k = 2
_, dense_indices = dense_index.search(query_embedding, k)
sparse_indices = np.argsort(bm25.get_scores(query.lower().split()))[::-1][:k]

# 5. RRF (Reciprocal Rank Fusion) to combine Dense and Sparse ranks
def reciprocal_rank_fusion(dense_res, sparse_res, k_val=60):
    fusion_scores = {}
    
    # Process Dense ranks
    for rank, doc_idx in enumerate(dense_res):
        fusion_scores[doc_idx] = fusion_scores.get(doc_idx, 0.0) + (1.0 / (k_val + rank + 1))
        
    # Process Sparse ranks
    for rank, doc_idx in enumerate(sparse_res):
        fusion_scores[doc_idx] = fusion_scores.get(doc_idx, 0.0) + (1.0 / (k_val + rank + 1))
        
    # Sort by highest RRF score
    sorted_docs = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_docs

# Combine ranks using RRF
ranked_results = reciprocal_rank_fusion(dense_indices[0], sparse_indices)
best_doc_idx = ranked_results[0][0]

# 6. Print final best match
print(f"Query: {query}")
print("Top Hybrid Match:", docs[best_doc_idx])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: FAISS search
Top Hybrid Match: FAISS vector search indexing


In [6]:
# Question - 4 :Implement a Multi-Turn Conversational Chatbot using Gemini API that maintains context across multiple user interactions.

In [9]:
chat = client.chats.create(model="gemini-3.6-flash")
response_5 = chat.send_message("Hello, my name is Suresh.")
print("User: Hello, my name is Suresh.")
print(f"Gemini: {response_5.text}\n")

response_6 = chat.send_message("What is my name?")
print("User: What is my name?")
print(f"Gemini: {response_6.text}")

User: Hello, my name is Suresh.
Gemini: Hello Suresh! It's nice to meet you. How can I help you today?

User: What is my name?
Gemini: Your name is Suresh!


In [10]:
# Question - 5 :Implement a FAISS-based retrieval-augmented generation (RAG) system using the Gemini API, where retrieved documents are used as context to generate more accurate and relevant responses.

In [11]:
docs = [
    "FAISS (Facebook AI Similarity Search) enables efficient similarity search and dense vector clustering.",
    "Gemini API provides state-of-the-art generative AI models to create context-aware responses.",
    "RAG (Retrieval-Augmented Generation) combines vector search retrieval with LLM generation for factual accuracy."
]

# 4. Prepare and encode documents for FAISS index (float32 required)
doc_embeddings = model.encode(docs).astype(np.float32)
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

# 5. Define query and retrieve the most relevant context
query = "How does FAISS work?"
query_embedding = model.encode([query]).astype(np.float32)

# Retrieve top-1 most relevant document
k = 1
distances, indices = index.search(query_embedding, k)
retrieved_idx = indices[0][0]
context = docs[retrieved_idx]

# 6. Construct prompt using the retrieved context
prompt = f"""Context:
{context}

Question: {query}
Instruction: Answer the question accurately using only the provided context."""

# 7. Generate response using Gemini API
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

# 8. Print Results
print(f"Query: {query}")
print(f"Retrieved Context: {context}\n")
print(f"Gemini Response:\n{response.text}")

Query: How does FAISS work?
Retrieved Context: FAISS (Facebook AI Similarity Search) enables efficient similarity search and dense vector clustering.

Gemini Response:
Based on the provided context, FAISS works by enabling efficient similarity search and dense vector clustering.


In [12]:
#------------------Set question 2------------------

In [13]:
# Question 6 : You are trying to use the T5 model to perform English-to-French translation. The model expects tokenized input.

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

input_text = "translate English to French: How are you doing today?"

inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)

outputs = model.generate(
    **inputs, 
    max_length=40, 
    num_beams=4,      # Beam search for better translation quality
    early_stopping=True
)


translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)


print("Input:", input_text)
print("Translation:", translated_text)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

C:\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

C:\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Input: translate English to French: How are you doing today?
Translation: Comment faites-vous aujourd'hui?


In [15]:
# Question 7: You wrote a RAG-like prompt flow but skipped the vector retrieval step. Complete it.

In [19]:
import os
import faiss
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from google import genai


load_dotenv()


client = genai.Client()


embed_model = SentenceTransformer("all-MiniLM-L6-v2")


docs = [
    "FAISS indexes vectors for fast similarity search.",
    "Gemini generates accurate text responses from context."
]


doc_embeddings = embed_model.encode(docs).astype(np.float32)


dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)


query = "How does FAISS work?"
query_embedding = embed_model.encode([query]).astype(np.float32)


distances, indices = index.search(query_embedding, k=1)
retrieved_context = docs[indices[0][0]]


final_response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=f"Context: {retrieved_context}\n\nQuestion: {query}"
)


print(f"Query: {query}")
print(f"Retrieved Context: {retrieved_context}\n")
print(f"Gemini Response:\n{final_response.text}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: How does FAISS work?
Retrieved Context: FAISS indexes vectors for fast similarity search.

Gemini Response:
At a high level, **FAISS (Facebook AI Similarity Search)** works by replacing slow, exhaustive searching with **Approximate Nearest Neighbor (ANN)** algorithms. 

Instead of comparing a query vector to every single vector in a database (which becomes impossible at a scale of millions or billions of vectors), FAISS organizes, compresses, and structures vectors into **indexes** to return results in milliseconds.

Here is a step-by-step breakdown of how FAISS works:

---

### 1. Vector Input and Distance Metrics
FAISS takes high-dimensional vectors (e.g., embeddings generated by AI models like BERT, OpenAI, or ResNet) and compares them using standard mathematical distance metrics:
* **L2 (Euclidean Distance):** Measures the direct distance between vector endpoints.
* **Inner Product (IP):** Measures vector alignment (equivalent to **Cosine Similarity** if vectors are normaliz

In [18]:
#!pip install sentence-transformers faiss-cpu google-genai python-dotenv numpy

In [20]:
# Question 8: You are trying to pass plain text to a model without tokenizing. Fix it to generate correct output using the Hugging Face T5 model.

In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


input_text = "translate English to French: How are you doing today?"


inputs = tokenizer(input_text, return_tensors="pt")


outputs = model.generate(
    **inputs, 
    max_new_tokens=40, 
    num_beams=4, 
    early_stopping=True
)


translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)


print("Input:", input_text)
print("Translation:", translated_text)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Input: translate English to French: How are you doing today?
Translation: Comment faites-vous aujourd'hui?


In [ ]:
#Question 9:What is the correct code to summarize a paragraph using a pre-trained BART model from Hugging Face?

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


text = """
The BART model is a sequence-to-sequence model built with a bidirectional encoder 
and an autoregressive decoder. It is effective for text generation, translation, 
and text summarization tasks.
"""

inputs = tokenizer(
    text, 
    return_tensors="pt", 
    max_length=1024, 
    truncation=True, 
    padding=True
)

summary_ids = model.generate(
    **inputs,
    max_length=50,
    min_length=10,
    num_beams=4,
    do_sample=False,
    early_stopping=True
)

summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)


print("Original Text Length:", len(text))
print("\nGenerated Summary:", summary_text)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

C:\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
from transformers import pipeline


generator = pipeline("text-generation", model="sshleifer/tiny-gpt2")


result = generator(
    "Once upon a time", 
    max_new_tokens=20,      # max_length-ku badhula max_new_tokens use panradhu safe
    do_sample=False,
    pad_token_id=50256      # GPT-2-oda EOS/Pad token ID to avoid warnings
)


print("Generated Text:", result[0]["generated_text"])